In [1]:
! pip install pandas matplotlib seaborn
import pandas as pd
import glob
import os
import json
import re
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [7]:
standard_confidence_metrics = [
    "mean_full",
    "median_full",
    "variance_full",
    "gap_full",
    'gap_probs_full',
    #"distinctiveness_full",
    'entropy_full',
    #'fisher_rao_distance_full',
    #'kl_divergence_full',
    #'inverse_probability_full',
    #'perplexity_full',
    #'renyi_divergence_full',
    'top_prob_full'
    ]

tail_confidence_metrics = [
    "mean_tail",
    "median_tail",
    "variance_tail",
    "gap_tail",
    "gap_probs_tail",
    #"distinctiveness_tail",
    'entropy_tail',
    #'fisher_rao_distance_tail',
    #'kl_divergence_tail',
    #'inverse_probability_tail',
    #'perplexity_tail',
    #'renyi_divergence_tail',
    'top_prob_tail'
    ]

group_lowest_metrics = [
    'mean_group_lowest',
    'median_group_lowest',
    'variance_group_lowest',
    'gap_group_lowest',
    'gap_probs_group_lowest',
    #'distinctiveness_group_lowest',
    'entropy_group_lowest',
    #'fisher_rao_distance_group_lowest',
    #'kl_divergence_group_lowest',
    #'inverse_probability_group_lowest',
    #'perplexity_group_lowest',
    #'renyi_divergence_group_lowest',
    #'top_prob_group_lowest'
]

group_highest_metrics = [
    'mean_group_highest',
    'median_group_highest',
    'variance_group_highest',
    'gap_group_highest',
    'gap_probs_group_highest',
    #'distinctiveness_group_highest',
    'entropy_group_highest',
    #'fisher_rao_distance_group_highest',
    #'kl_divergence_group_highest',
    #'inverse_probability_group_highest',
    #'perplexity_group_highest',
    #'renyi_divergence_group_highest',
    #'top_prob_group_highest'
]

group_bottom_metrics = [
    'mean_group_bottom_pct',
    'median_group_bottom_pct',
    'variance_group_bottom_pct',
    'gap_group_bottom_pct',
    'gap_probs_group_bottom_pct',
    #'distinctiveness_group_bottom_pct',
    'entropy_group_bottom_pct',
    #'fisher_rao_distance_group_bottom_pct',
    #'kl_divergence_group_bottom_pct',
    #'inverse_probability_group_bottom_pct',
    #'perplexity_group_bottom_pct',
    #'renyi_divergence_group_bottom_pct',
    #'top_prob_group_bottom_pct'
]

group_top_metrics = [
    'mean_group_top_pct',
    'median_group_top_pct',
    'variance_group_top_pct',
    'gap_group_top_pct',
    'gap_probs_group_top_pct',
    #'distinctiveness_group_top_pct',
    'entropy_group_top_pct',
    #'fisher_rao_distance_group_top_pct',
    #'kl_divergence_group_top_pct',
    #'inverse_probability_group_top_pct',
    #'perplexity_group_top_pct',
    #'renyi_divergence_group_top_pct',
    #'top_prob_group_top_pct'
]

base_metrics = [
    'mean',
    'median',
    'variance',
    'gap',
    'gap_probs',
    #'distinctiveness',
    'entropy',
    #'fisher_rao_distance',
    #'kl_divergence',
    #'inverse_probability',
    #'perplexity',
    #'renyi_divergence',
    #'top_prob'
]

In [9]:
#!/usr/bin/env python3
import json
import os
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

# ── config ────────────────────────────────────────────────────────────────────
models   = ["qwen1_7b"]
datasets = ["kodcode", "humaneval"]
turns    = [1,2,3]
base_dir = "/efs/cactts/data"

# ── load data ─────────────────────────────────────────────────────────────────
def load_records(model, dataset, turns, base_dir, ref=False):
    records = []
    if ref:
        metrics_name = "ref_response_metrics.jsonl"
        subdir = 'ref_correct'
        base_dir = "/efs/cactts/ref_data"
    else:
        metrics_name = "all_response_metrics.jsonl"
        subdir = ''
    for turn in turns:
        metrics_file = os.path.join(base_dir, model, dataset, f"turn{turn}/{subdir}",  metrics_name)
        print(f"Looking for {metrics_file}...")
        if not os.path.exists(metrics_file):
            print(f"Skipping {model}/{dataset}/turn{turn} — not found")
            continue
        with open(metrics_file, 'r') as f:
            for line in f:
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError:
                    continue
    return records

# ── detect confidence metrics ─────────────────────────────────────────────────
def get_confidence_metrics(records):
    """Find all numeric fields that look like confidence metrics."""
    if not records:
        return []
    sample = records[0]
    skip = {'problem_id', 'rollout_idx', 'is_correct', 'response_text', 'expected_answer'}
    metrics = []
    for k, v in sample.items():
        if k not in skip and (k in standard_confidence_metrics or k in tail_confidence_metrics) and isinstance(v, (int, float)) and v is not None:
            metrics.append(k)
    return metrics

# ── plot ──────────────────────────────────────────────────────────────────────
def plot_histograms(model, dataset, records, confidence_metrics, out_dir, ref=False):
    if not records or not confidence_metrics:
        print(f"No data to plot for {model}/{dataset}")
        return

    correct   = [r for r in records if r.get('is_correct')]
    incorrect = [r for r in records if not r.get('is_correct')]
    print(f"{model}/{dataset}: {len(correct)} correct, {len(incorrect)} incorrect")

    n_metrics = len(confidence_metrics)
    ncols = 3
    nrows = (n_metrics + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows))
    axes = np.array(axes).flatten() if n_metrics > 1 else [axes]

    for ax, metric in zip(axes, confidence_metrics):
        correct_vals   = [r[metric] for r in correct   if r.get(metric) is not None]
        incorrect_vals = [r[metric] for r in incorrect if r.get(metric) is not None]

        all_vals = correct_vals + incorrect_vals
        if not all_vals:
            ax.set_visible(False)
            continue

        bins = np.linspace(min(all_vals), max(all_vals), 40)

        ax.hist(correct_vals,   bins=bins, alpha=0.6, color='green', label=f'Correct (n={len(correct_vals)})',  )
        ax.hist(incorrect_vals, bins=bins, alpha=0.6, color='red',   label=f'Incorrect (n={len(incorrect_vals)})')

        ax.set_title(metric, fontsize=12)
        ax.set_xlabel('Confidence')
        ax.set_ylabel('Count')
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)

    # hide unused axes
    for ax in axes[n_metrics:]:
        ax.set_visible(False)

    fig.suptitle(f"{model} — {dataset} (turns {turns})", fontsize=14, fontweight='bold')
    plt.tight_layout()

    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"{model}_{dataset}_confidence_histograms{'_org' if not ref else '_ref'}.png")
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved → {out_path}")

# ── main ──────────────────────────────────────────────────────────────────────
# def main():
#     out_dir = "./plots"
# 
#     for model in models:
#         for dataset in datasets:
#             print(f"\nLoading {model}/{dataset}...")
#             records = load_records(model, dataset, turns, base_dir)
#             print(f"  Loaded {len(records)} records")
# 
#             confidence_metrics = get_confidence_metrics(records)
#             print(f"  Confidence metrics found: {confidence_metrics}")
# 
#             plot_histograms(model, dataset, records, confidence_metrics, out_dir)
# 
# if __name__ == "__main__":
#     main()

In [10]:
out_dir = "./plots"
ref = True
for model in models:
    for dataset in datasets:
        print(f"\nLoading {model}/{dataset}...")
        records = load_records(model, dataset, turns, base_dir, ref)
        print(f"  Loaded {len(records)} records")

        confidence_metrics = get_confidence_metrics(records)
        print(f"  Confidence metrics found: {confidence_metrics}")

        plot_histograms(model, dataset, records, confidence_metrics, out_dir, ref)


Loading qwen1_7b/kodcode...
Looking for /efs/cactts/ref_data/qwen1_7b/kodcode/turn1/ref_correct/ref_response_metrics.jsonl...


Looking for /efs/cactts/ref_data/qwen1_7b/kodcode/turn2/ref_correct/ref_response_metrics.jsonl...
Looking for /efs/cactts/ref_data/qwen1_7b/kodcode/turn3/ref_correct/ref_response_metrics.jsonl...
  Loaded 48000 records
  Confidence metrics found: ['mean_full', 'mean_tail', 'median_full', 'median_tail', 'variance_full', 'variance_tail', 'gap_full', 'gap_tail', 'top_prob_full', 'top_prob_tail', 'entropy_full', 'entropy_tail', 'gap_probs_full', 'gap_probs_tail']
qwen1_7b/kodcode: 28643 correct, 19357 incorrect
Saved → ./plots/qwen1_7b_kodcode_confidence_histograms_ref.png

Loading qwen1_7b/humaneval...
Looking for /efs/cactts/ref_data/qwen1_7b/humaneval/turn1/ref_correct/ref_response_metrics.jsonl...
Looking for /efs/cactts/ref_data/qwen1_7b/humaneval/turn2/ref_correct/ref_response_metrics.jsonl...
Looking for /efs/cactts/ref_data/qwen1_7b/humaneval/turn3/ref_correct/ref_response_metrics.jsonl...
  Loaded 7872 records
  Confidence metrics found: ['mean_full', 'mean_tail', 'median_full', '

In [1]:
import pickle


In [5]:
with open('../cache_logprobs/gptoss_humaneval_turn1_original.pkl', 'rb') as f:
    data = pickle.load(f)
    
with open('../cache_logprobs/gptoss_humaneval_turn1_ref.pkl', 'rb') as f:
    data2 = pickle.load(f)

In [ ]:
data[0]

In [ ]:
data2

In [ ]:
"""
Logprob Analysis: Correct vs Incorrect Rollout Distributions
Usage:
    python logprob_analysis.py --model gptoss --dataset humaneval [--cache_dir ../cache_logprobs]
"""

import pickle
import os
import argparse
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path


# ──────────────────────────────────────────────
# I/O helpers
# ──────────────────────────────────────────────

def load_pkl(cache_dir: str, model: str, dataset: str, turn: int, variant: str) -> list:
    """Load a single pkl file.  variant is 'original' or 'ref'."""
    fname = f"{model}_{dataset}_turn{turn}_{variant}.pkl"
    fpath = os.path.join(cache_dir, fname)
    with open(fpath, "rb") as f:
        return pickle.load(f)


# ──────────────────────────────────────────────
# Metric computation
# ──────────────────────────────────────────────

def compute_metric(logprobs: np.ndarray) -> float:
    """Aggregate per-token logprobs into a single scalar.
    Currently: mean of all token logprobs (i.e. average log-likelihood).
    Swap this function to change the metric globally.
    """
    arr = np.asarray(logprobs, dtype=np.float32)
    return float(np.mean(arr))


# ──────────────────────────────────────────────
# Data extraction
# ──────────────────────────────────────────────

def extract_metrics(records: list) -> tuple[list[float], list[float]]:
    """Return (correct_metrics, incorrect_metrics) from a list of rollout records."""
    correct, incorrect = [], []
    for rec in records:
        lp = rec.get("logprobs")
        if lp is None:
            continue
        metric = compute_metric(lp)
        if rec.get("is_correct"):
            correct.append(metric)
        else:
            incorrect.append(metric)
    return correct, incorrect


def collect_all_turns(
    cache_dir: str, model: str, dataset: str, turns: list[int], variant: str
) -> tuple[list[float], list[float]]:
    """Pool correct / incorrect metrics across all requested turns for one variant."""
    all_correct, all_incorrect = [], []
    for t in turns:
        try:
            records = load_pkl(cache_dir, model, dataset, t, variant)
        except FileNotFoundError as e:
            print(f"  [warn] {e} — skipping")
            continue
        c, inc = extract_metrics(records)
        all_correct.extend(c)
        all_incorrect.extend(inc)
        print(f"  turn {t} ({variant}): {len(c)} correct, {len(inc)} incorrect")
    return all_correct, all_incorrect


# ──────────────────────────────────────────────
# Plotting: distribution (original / ref)
# ──────────────────────────────────────────────

def _plot_distribution_ax(ax, correct: list[float], incorrect: list[float], title: str) -> None:
    """Draw overlapping histograms of correct (green) and incorrect (red) on ax."""
    bins = 40

    if incorrect:
        ax.hist(incorrect, bins=bins, color="red",   alpha=0.55, label=f"Incorrect (n={len(incorrect)})", density=True)
    if correct:
        ax.hist(correct,   bins=bins, color="green", alpha=0.55, label=f"Correct   (n={len(correct)})",   density=True)

    # Vertical mean lines
    if incorrect:
        ax.axvline(np.mean(incorrect), color="darkred",   linestyle="--", linewidth=1.4,
                   label=f"Mean incorrect = {np.mean(incorrect):.3f}")
    if correct:
        ax.axvline(np.mean(correct),   color="darkgreen", linestyle="--", linewidth=1.4,
                   label=f"Mean correct   = {np.mean(correct):.3f}")

    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("Mean log-prob per rollout", fontsize=11)
    ax.set_ylabel("Density", fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.3)


def plot_distributions(
    cache_dir: str, model: str, dataset: str, turns: list[int], save: bool = True
) -> None:
    """One figure with two subplots: _original (left) and _ref (right)."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(
        f"Logprob distribution — {model} / {dataset}  (turns {turns})",
        fontsize=14, fontweight="bold", y=1.01,
    )

    for ax, variant in zip(axes, ["original", "ref"]):
        print(f"\nCollecting '{variant}' data …")
        correct, incorrect = collect_all_turns(cache_dir, model, dataset, turns, variant)
        _plot_distribution_ax(
            ax, correct, incorrect,
            title=f"variant = {variant}"
        )

    plt.tight_layout()
    if save:
        out = f"logprob_dist_{model}_{dataset}.png"
        fig.savefig(out, dpi=150, bbox_inches="tight")
        print(f"\n[saved] {out}")
    plt.show()


# ──────────────────────────────────────────────
# Plotting: ratio original / ref
# ──────────────────────────────────────────────

def plot_ratio(
    cache_dir: str, model: str, dataset: str, turns: list[int], save: bool = True
) -> None:
    """Plot the ratio of original-metric to ref-metric per rollout (matched by index).

    For each rollout that appears in both files we compute:
        ratio = metric_original / metric_ref
    and separate into correct / incorrect bins.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
    fig.suptitle(
        f"Logprob ratio  (original / ref) — {model} / {dataset}  (turns {turns})",
        fontsize=14, fontweight="bold", y=1.01,
    )

    ratio_correct_all, ratio_incorrect_all = [], []

    for t in turns:
        try:
            orig_records = load_pkl(cache_dir, model, dataset, t, "original")
            ref_records  = load_pkl(cache_dir, model, dataset, t, "ref")
        except FileNotFoundError as e:
            print(f"  [warn] {e} — skipping turn {t}")
            continue

        n = min(len(orig_records), len(ref_records))
        for o, r in zip(orig_records[:n], ref_records[:n]):
            lp_o = o.get("logprobs")
            lp_r = r.get("logprobs")
            if lp_o is None or lp_r is None:
                continue
            m_o = compute_metric(lp_o)
            m_r = compute_metric(lp_r)
            if m_r == 0:
                continue
            ratio = m_o / m_r
            if o.get("is_correct"):
                ratio_correct_all.append(ratio)
            else:
                ratio_incorrect_all.append(ratio)

    bins = 40

    # Left: overlapping histograms
    ax = axes[0]
    if ratio_incorrect_all:
        ax.hist(ratio_incorrect_all, bins=bins, color="red",   alpha=0.55,
                label=f"Incorrect (n={len(ratio_incorrect_all)})", density=True)
    if ratio_correct_all:
        ax.hist(ratio_correct_all,   bins=bins, color="green", alpha=0.55,
                label=f"Correct   (n={len(ratio_correct_all)})",   density=True)
    for vals, col in [(ratio_incorrect_all, "darkred"), (ratio_correct_all, "darkgreen")]:
        if vals:
            ax.axvline(np.mean(vals), color=col, linestyle="--", linewidth=1.4)
    ax.axvline(1.0, color="black", linestyle=":", linewidth=1.2, label="ratio = 1")
    ax.set_title("Overlapping distributions", fontsize=12)
    ax.set_xlabel("Ratio  (orig metric / ref metric)", fontsize=11)
    ax.set_ylabel("Density", fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.3)

    # Right: box plot comparison
    ax2 = axes[1]
    data_to_plot = []
    labels = []
    colors = []
    if ratio_incorrect_all:
        data_to_plot.append(ratio_incorrect_all)
        labels.append("Incorrect")
        colors.append("red")
    if ratio_correct_all:
        data_to_plot.append(ratio_correct_all)
        labels.append("Correct")
        colors.append("green")

    if data_to_plot:
        bp = ax2.boxplot(data_to_plot, labels=labels, patch_artist=True, notch=False,
                         medianprops=dict(color="black", linewidth=2))
        for patch, col in zip(bp["boxes"], colors):
            patch.set_facecolor(col)
            patch.set_alpha(0.6)
    ax2.axhline(1.0, color="black", linestyle=":", linewidth=1.2, label="ratio = 1")
    ax2.set_title("Box plot of ratios", fontsize=12)
    ax2.set_ylabel("Ratio  (orig / ref)", fontsize=11)
    ax2.legend(fontsize=9)
    ax2.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    if save:
        out = f"logprob_ratio_{model}_{dataset}.png"
        fig.savefig(out, dpi=150, bbox_inches="tight")
        print(f"\n[saved] {out}")
    plt.show()


# ──────────────────────────────────────────────
# CLI entry-point
# ──────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(description="Logprob distribution analyser")
    parser.add_argument("--model",     default="gptoss",     help="Model name prefix")
    parser.add_argument("--dataset",   default="humaneval",  help="Dataset name")
    parser.add_argument("--cache_dir", default="../cache_logprobs", help="Directory with pkl files")
    parser.add_argument("--turns",     nargs="+", type=int, default=[1, 2, 3])
    parser.add_argument("--no_save",   action="store_true",  help="Don't save figures to disk")
    args = parser.parse_args()

    save = not args.no_save

    print("=== Distribution plots ===")
    plot_distributions(args.cache_dir, args.model, args.dataset, args.turns, save=save)

    print("\n=== Ratio plots ===")
    plot_ratio(args.cache_dir, args.model, args.dataset, args.turns, save=save)

plot_distributions('../cache_logprobs', 'gptoss', 'humaneval',[1,2,3], save=True)

In [8]:
plot_distributions('../cache_logprobs', 'gptoss', 'humaneval',[1,2,3], save=True)

NameError: name 'gptoss' is not defined